# 📦 Model prefetch to Google Drive (bypass a flaky local network)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vedmalex/higgs-local-test/blob/main/notebooks/model_prefetch_to_drive.ipynb)

Downloads models listed in `notebooks/model_catalog.json` (the project's single inventory of every model it uses, has used, or has considered, each with a `fetch_now` flag) using Colab's network (fast, stable) instead of a local machine's -- useful when the local network (e.g. a VPN reporting `Transient Connection`) keeps stalling multi-gigabyte Hugging Face downloads. Packs each model into a single `.tar` per repo (preserving the Hugging Face cache's internal blob/snapshot symlink layout -- Drive's FUSE mount does not reliably preserve symlinks on its own, so they must travel inside an archive, not as loose files) and copies it to Google Drive.

**No GPU needed** -- this notebook only downloads and packs files, so pick a CPU-only runtime to avoid burning GPU quota on a pure I/O task.

## After this notebook finishes

On the local machine, for each archive downloaded from `MyDrive/higgs-benchmark/model-cache/`:

```bash
tar -xf ~/Downloads/<name>.tar -C ~/.cache/huggingface/hub/
```

That's it -- `huggingface_hub` recognizes the extracted `models--<org>--<name>` directory as already cached (same revision, same blob hashes) and every later `make tts` / `make stt` / Qwen local run skips downloading it again.


## 1. Google Drive workspace

Same pattern as the other notebooks in this repo: results (here, the archives) go to Drive as each one finishes, and the VM disconnects unconditionally at the end.


In [ ]:
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

MODEL_CACHE_DIR = Path("/content/drive/MyDrive/higgs-benchmark/model-cache")
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Archives will be written to: {MODEL_CACHE_DIR}")


## 2. Choose which models to fetch

The model list lives in `model_catalog.json`, not in this notebook -- it is the single inventory of every model this project uses, has used, or has considered (state, size, license, and a `fetch_now` flag per entry; see the file's own `_readme` field and AGENTS.md's "Model downloads" rule for how to add an entry). This cell fetches that JSON from the repo and queues only the entries with `fetch_now: true`. To add or drop a model from a run, edit `fetch_now` in `model_catalog.json` and commit it -- don't hand-edit a Python list here.

In [ ]:
import json
import urllib.request

# Single source of truth for the model list -- see notebooks/model_catalog.json in the repo.
# Fetched from `main` (same branch the "Open in Colab" badge above points at) rather than
# duplicated inline, so editing the catalog file is enough; this notebook never needs a
# matching edit just to add or drop a model.
CATALOG_URL = (
    "https://raw.githubusercontent.com/vedmalex/higgs-local-test/main/"
    "notebooks/model_catalog.json"
)
with urllib.request.urlopen(CATALOG_URL) as resp:
    catalog = json.loads(resp.read())["models"]

MODELS = [
    (m["name"], m["repo_id"], m["revision"], m["allow_patterns"])
    for m in catalog
    if m["fetch_now"]
]
for name, repo_id, _, _ in MODELS:
    assert repo_id, f"{name}: fetch_now=true but repo_id is not pinned yet in model_catalog.json"

print(f"{len(MODELS)} models queued (fetch_now=true in model_catalog.json):")
for name, repo_id, revision, allow_patterns in MODELS:
    extra = f" (allow_patterns={allow_patterns})" if allow_patterns else ""
    print(f"  - {name}: {repo_id}" + (f"@{revision}" if revision else "") + extra)

## 3. Download each model, pack it, copy to Drive

Downloads into the VM's local disk (`HF_HOME` below) -- fast, and Colab's own network is what we're relying on being stable, not FUSE-backed Drive I/O for the download itself. Only the final archive crosses onto Drive.

Each model is archived and uploaded **immediately after it finishes**, not batched at the end: if the notebook is interrupted partway through the model list, everything already done is already safe on Drive.


In [ ]:
import os
import subprocess
import tarfile
import time
from pathlib import Path

HF_HOME = Path("/content/hf-cache")
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_DISABLE_XET"] = "1"  # matches scripts/download_models.sh

from huggingface_hub import snapshot_download


def download_with_retry(repo_id, revision, allow_patterns, max_attempts=10):
    for attempt in range(1, max_attempts + 1):
        try:
            return snapshot_download(repo_id, revision=revision, allow_patterns=allow_patterns)
        except Exception as error:
            if attempt == max_attempts:
                raise
            backoff = min(attempt * 5, 30)
            print(f"  attempt {attempt}/{max_attempts} failed ({error!r}); retrying in {backoff}s")
            time.sleep(backoff)


def is_locally_complete(model_dir: Path) -> bool:
    """True if model_dir exists and has no leftover .incomplete blobs -- i.e. a prior cell
    run (or an earlier attempt in this same VM session) already finished downloading it, so
    packing can skip straight past a fresh snapshot_download call entirely."""
    if not model_dir.is_dir():
        return False
    blobs = model_dir / "blobs"
    if not blobs.is_dir():
        return False
    return not any(blobs.glob("*.incomplete"))


# List once, fresh, rather than checking archive_path.exists() per model -- a single
# directory listing avoids relying on any per-path caching the Drive FUSE mount might do
# and makes exactly what's already on Drive explicit before deciding what to skip.
existing_archives = {p.name for p in MODEL_CACHE_DIR.glob("*.tar")}
print(f"Already on Drive: {sorted(existing_archives) or '(none)'}")

results = []
for name, repo_id, revision, allow_patterns in MODELS:
    print(f"\n=== {name} ({repo_id}) ===")
    archive_name = f"{name}.tar"
    archive_path = MODEL_CACHE_DIR / archive_name
    if archive_name in existing_archives:
        print(f"already on Drive: {archive_path} ({archive_path.stat().st_size / 1e9:.2f} GB) -- skipping")
        results.append((name, "SKIPPED (already on Drive)", archive_path))
        continue

    safe_repo = repo_id.replace("/", "--")
    model_dir = HF_HOME / "hub" / f"models--{safe_repo}"

    try:
        if is_locally_complete(model_dir):
            # Already downloaded in this VM session (e.g. a previous cell run got the
            # weights but was interrupted before packing/upload) -- go straight to
            # packing, no network call needed.
            print(f"already downloaded locally at {model_dir} -- skipping snapshot_download, packing directly")
        else:
            download_with_retry(repo_id, revision, allow_patterns)
        assert model_dir.is_dir(), f"expected cache dir missing: {model_dir}"

        local_tar = Path("/content") / archive_name
        with tarfile.open(local_tar, "w") as tf:
            tf.add(model_dir, arcname=model_dir.name)
        print(f"packed {local_tar} ({local_tar.stat().st_size / 1e9:.2f} GB)")

        subprocess.run(["cp", str(local_tar), str(archive_path)], check=True)
        local_tar.unlink()  # free VM disk once the Drive copy exists
        existing_archives.add(archive_name)
        print(f"uploaded to {archive_path}")
        results.append((name, "OK", archive_path))
    except Exception as error:
        print(f"FAILED: {error!r}")
        results.append((name, f"FAILED: {error!r}", None))

print("\n=== Summary ===")
for name, status, path in results:
    print(f"{name}: {status}" + (f" -> {path}" if path else ""))


## 4. Instructions for the local machine

Download each `.tar` from `MyDrive/higgs-benchmark/model-cache/` (Drive web UI, or let Drive desktop sync it automatically), then, in a terminal:

```bash
for f in ~/Downloads/*.tar; do
  tar -xf "$f" -C ~/.cache/huggingface/hub/
done
```

Verify with `du -sh ~/.cache/huggingface/hub/models--*` -- sizes should match what this notebook printed per model. `make tts` / `make stt` / the Qwen local runners will then find everything already cached and skip downloading.


## 5. Завершение: безусловное отключение


In [ ]:
try:
    print(f"На Диске сохранено: {MODEL_CACHE_DIR}")
    for artefact in sorted(MODEL_CACHE_DIR.iterdir()):
        if artefact.is_file():
            print(f"  - {artefact.name} ({artefact.stat().st_size / 1e9:.2f} GB)")
except Exception as error:
    print(f"не удалось перечислить артефакты: {error!r}")

try:
    from google.colab import drive
    drive.flush_and_unmount()
    print("💾 Данные синхронизированы: MyDrive/higgs-benchmark/model-cache/")
finally:
    from google.colab import runtime
    print("🛑 Отключение ВМ...")
    runtime.unassign()
